# V2a-RSN 220127_F4_run2 — Manifold Notebook: Partial Correlation Baseline

Prerequisites: the dataset `shared/recording` checkpoint and the `correlation_partial` connectivity stage for the selected run.
Parallel lane: yes, once the method-specific connectivity stage exists.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused checkpoints are reported explicitly.


In [ ]:
import sys
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]


In [ ]:
from omegaconf import OmegaConf

from effectome.manifold import ManifoldConfig
from notebooks._shared import PROJECT_ROOT, make_run, print_stage_status, run_manifold

DATASET_ID = "220127_F4_run2"
DATASET_OUTPUT_ID = "v2a-rsns"
RECORDING_ID = "220127_F4_run2"
RUN_ID = "reference"
METHOD = "correlation_partial"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
RECORDING_PATH = OUTPUT_ROOT / RUN_ID / "shared" / "stages" / "recording" / "artifact.pkl"
MANIFOLD_CONFIG_NAME = "classical"
MANIFOLD_CONFIG_PATH = PROJECT_ROOT / "conf" / "manifold" / f"{MANIFOLD_CONFIG_NAME}.yaml"
MANIFOLD_BEHAVIOR_KEY = "continuous"
FORCE = False

MANIFOLD_YAML = OmegaConf.to_container(OmegaConf.load(MANIFOLD_CONFIG_PATH), resolve=True)
if not isinstance(MANIFOLD_YAML, dict):
    raise TypeError(f"Expected a mapping in {MANIFOLD_CONFIG_PATH}")
MANIFOLD_CFG = ManifoldConfig(
    **{**MANIFOLD_YAML, "behavior_key": MANIFOLD_BEHAVIOR_KEY}
)
LINKING_CFG = {
    "embargo": 4,
    "group_by": "recording",
    "lag_extension": 0,
    "preprocessing_past_support": 0,
    "preprocessing_future_support": 0,
}

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_stage_status(run)


In [ ]:
manifold = run_manifold(
    run,
    recording_path=RECORDING_PATH,
    manifold_cfg=MANIFOLD_CFG,
    linking_cfg=LINKING_CFG,
    force=FORCE,
)
print(manifold.window_embedding.shape)
print(manifold.metadata["window_embedding_mode"])
print_stage_status(run)
